In [1]:
import numpy as np  # type: ignore
import pandas as pd  # type: ignore

import fiona  # type: ignore
from fiona.crs import from_epsg # type: ignore
import geopandas as gpd  # type: ignore
import shapely # type: ignore
from shapely.geometry import Point

import matplotlib.pyplot as plt  # type: ignore
import seaborn as sns  # type: ignore


%matplotlib inline

In [2]:
#Read shp file

estsoil_12c = "Data\EstSoil-EH_v1.2c.shp\EstSoil-EH_v1.2c.shp"
data = gpd.read_file(estsoil_12c,encoding='utf-8')

display(data.head())

,orig_fid,est_soilty,wrb_code,wrb_main,est_txcode,nlayers,zmx,z1,est_txt1,lxtype1,...,grassland_,area_wetla,wetland_pc,area_urban,urban_pct,area_water,water_pct,area_other,other_pct,geometry
0,0,Ag,FL-gln,FL,l,1.0,1000.0,1000.0,l,S,...,0.000000,0.0,0.0,0.000000,0.000000,0.066504,0.003678,1808.194742,99.996322,"POLYGON ((698614.390 6447795.940, 698612.523 6..."
1,1,Ag,FL-gln,FL,l,1.0,1000.0,1000.0,l,S,...,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,"POLYGON ((668432.067 6543565.911, 668425.630 6..."
2,2,Ag,FL-gln,FL,l,1.0,1000.0,1000.0,l,S,...,71.058156,0.0,0.0,386.624055,14.876251,359.077009,13.816315,0.000000,0.000000,"POLYGON ((668550.130 6543573.200, 668543.495 6..."
3,3,Ag,FL-gln,FL,l,1.0,1000.0,1000.0,l,S,...,66.551261,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,"POLYGON ((679817.320 6545854.050, 679829.522 6..."
4,4,Ag,FL-gln,FL,l,1.0,1000.0,1000.0,l,S,...,98.280958,0.0,0.0,0.000000,0.000000,28.399424,1.719042,0.000000,0.000000,"POLYGON ((677362.640 6546501.230, 677345.370 6..."


In [3]:
#Drop soc1 soc2 soc3 soc4 values
no_soc_data = data.drop(["soc1","soc2","soc3","soc4"],axis=1)
no_soc_data.columns

Index(['orig_fid', 'est_soilty', 'wrb_code', 'wrb_main', 'est_txcode',
       'nlayers', 'zmx', 'z1', 'est_txt1', 'lxtype1', 'est_crs1', 'sand1',
       'silt1', 'clay1', 'rock1', 'bd1', 'k1', 'awc1', 'z2', 'est_txt2',
       'lxtype2', 'est_crs2', 'sand2', 'silt2', 'clay2', 'rock2', 'bd2', 'k2',
       'awc2', 'z3', 'est_txt3', 'lxtype3', 'est_crs3', 'sand3', 'silt3',
       'clay3', 'rock3', 'bd3', 'k3', 'awc3', 'z4', 'est_txt4', 'lxtype4',
       'est_crs4', 'sand4', 'silt4', 'clay4', 'rock4', 'bd4', 'k4', 'awc4',
       'unit_area', 'tri_mean', 'tri_stdev', 'tri_median', 'twi_mean',
       'twi_stdev', 'twi_median', 'slp_mean', 'slp_stdev', 'slp_median',
       'ls_mean', 'ls_stdev', 'ls_median', 'area_drain', 'drain_pct',
       'area_arabl', 'arable_pct', 'area_fores', 'forest_pct', 'area_grass',
       'grassland_', 'area_wetla', 'wetland_pc', 'area_urban', 'urban_pct',
       'area_water', 'water_pct', 'area_other', 'other_pct', 'geometry'],
      dtype='object')

In [5]:
#Drop all unnecessary columns(ones which are calculated based on previous SOC data and the ones which are already exist in soc_samples data)
#before joining to soc_samples data


# cleaned_data = no_soc_data.drop(['wrb_code','wrb_main', 'nlayers', 'est_txt1', 'lxtype1', 'est_crs1', 'bd1', 'est_txt2', 'lxtype2', 
#                     'est_crs2','bd2','k1','k2','est_txt3', 'lxtype3', 'est_crs3', 'bd3', 'k3', 'est_txt4','lxtype4','est_crs4',
#                        'bd4', 'k4','unit_area', 'tri_mean', 'tri_stdev', 'tri_median', 'twi_mean','twi_stdev', 'twi_median', 
#                        'slp_mean', 'slp_stdev', 'slp_median',  'ls_mean', 'ls_stdev', 'ls_median', 'area_drain', 'drain_pct',
#                        'area_arabl', 'arable_pct', 'area_fores', 'forest_pct', 'area_grass','grassland_', 'area_wetla', 
#                        'wetland_pc', 'area_urban', 'urban_pct','area_water', 'water_pct', 'area_other', 'other_pct', 'geometry',
#                        'awc1', 'awc2', 'awc3', 'awc4' ],axis=1)

cleaned_data.head()


,orig_fid,est_soilty,est_txcode,zmx,z1,sand1,silt1,clay1,rock1,z2,...,z3,sand3,silt3,clay3,rock3,z4,sand4,silt4,clay4,rock4
0,0,Ag,l,1000.0,1000.0,90,5,5,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Ag,l,1000.0,1000.0,90,5,5,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Ag,l,1000.0,1000.0,90,5,5,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Ag,l,1000.0,1000.0,90,5,5,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,Ag,l,1000.0,1000.0,90,5,5,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# read soc data
soc_samples = gpd.read_file('Data\data_deposit_supplements_1.2c\data_deposit_supplements_1.2c\soc_rf_model\soc_rf_redone_3301.gpkg', layer = "soc_train_samples_outrem_origfid_avg")
soc_samples.head()


,orig_fid,SOC_mean,SOC_std,SOC_med,SOC_min,SOC_max,origin,origin_p,s_count,upd_siffer,...,area_other,unit_area,arable_pct,forest_pct,grassland_pct,wetland_pct,urban_pct,water_pct,other_pct,geometry
0,15159,11.078491,0.000559,11.078491,11.078095,11.078886,estonian_alvars_soil_sample_single_dataset.shp,0.500000,2,ArG,...,20363.430457,89012.199332,0.209052,30.211757,45.509613,16.460368,0.000000,1.192456,22.877123,POINT (443390.670 6494246.123)
1,22283,1.056000,0.669724,0.840000,0.440000,2.190000,envir_biosoils_survey.shp,1.000000,5,D,...,0.000000,25583.014316,44.760985,55.239015,0.000000,0.000000,0.000000,0.000000,0.000000,POINT (661204.025 6433312.518)
2,45116,5.339205,NaN,5.339205,5.339205,5.339205,rmk_soil_data.shp,1.000000,1,Dg,...,0.000000,1948.785206,0.000000,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,POINT (593729.152 6457480.739)
3,86756,1.000000,0.000000,1.000000,1.000000,1.000000,envir_kese_muld_export.shp,1.000000,2,E2o,...,526.615939,62036.861342,82.160218,12.099221,0.000000,0.000000,4.891685,0.000000,0.848876,POINT (644275.296 6425833.836)
4,94817,4.029763,0.851630,3.514495,3.446287,5.128507,estonian_alvars_open_areas.shp,0.166667,6,Gh',...,149612.185454,354413.777500,0.515653,56.275738,0.693469,0.000000,0.035944,0.265209,42.213987,POINT (376404.901 6468198.723)


In [20]:
#merge dataframes of soc file and 
merged_data = pd.merge(cleaned_data, soc_samples, on= "orig_fid")
display(merged_data.sample(50))

,orig_fid,est_soilty,est_txcode,zmx,z1,sand1,silt1,clay1,rock1,z2,...,area_other,unit_area,arable_pct,forest_pct,grassland_pct,wetland_pct,urban_pct,water_pct,other_pct,geometry
408,601041,LP,sl40-70/ls₂,1000.0,550.0,82,9,9,0,1000.0,...,0.000000,5.078975e+04,0.000000,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,POINT (663776.502 6399418.730)
322,432408,LGn,pl,1000.0,1000.0,90,3,7,0,NaN,...,6664.647752,4.034168e+04,0.000000,81.223991,0.000000,0.000000,2.255509,0.000000,16.520500,POINT (561657.402 6508086.172)
201,302248,KIg,ls₁70/+ls₂,1000.0,700.0,65,20,15,0,1000.0,...,0.000000,2.015549e+05,0.000000,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,POINT (596547.517 6445786.089)
270,394662,Kr,ls₁10-25/r₄ls₁,1000.0,175.0,65,20,15,0,1000.0,...,95554.888400,4.223413e+06,3.493320,92.901816,0.970948,0.002706,0.375519,0.007071,2.262504,POINT (513245.739 6536889.544)
426,688884,M'',t₃50-100/ls₁,1000.0,750.0,15,15,70,0,1000.0,...,9075.643910,8.064329e+05,0.000000,84.857199,12.798887,1.212201,0.000000,0.006307,1.125406,POINT (697258.000 6457225.000)
200,298210,KIg,v⁰₁sl35/v⁰₁ls₁30-40/v₁ls₁,1000.0,350.0,82,9,9,6,700.0,...,8431.585060,5.376973e+05,86.385576,7.512886,1.877389,0.000000,1.469435,1.186624,1.568091,POINT (654570.272 6479316.878)
204,307554,KIg,v⁰₁ls₁55-90/v₁ls₁,1000.0,725.0,65,20,15,6,1000.0,...,493.784205,9.429586e+05,79.546790,16.404107,1.773416,0.000000,2.155093,0.068228,0.052365,POINT (650191.145 6473486.107)
446,726654,R''',t₁150,1500.0,1500.0,25,25,50,0,NaN,...,0.000000,2.755280e+06,0.000000,1.833028,0.000000,96.934699,0.000000,1.232274,0.000000,POINT (569286.545 6518205.859)
143,259284,Kh'',r₂sl25-28/p,1000.0,265.0,82,9,9,15,1000.0,...,194.471586,6.669386e+04,0.092075,87.054921,10.565128,0.000000,1.996287,0.000000,0.291588,POINT (476458.187 6490946.034)
118,246815,Kg,r₂sl30/r₄sl,1000.0,300.0,82,9,9,15,1000.0,...,2668.642048,3.406736e+05,1.828750,95.641061,1.103003,0.643843,0.000000,0.000000,0.783343,POINT (377199.969 6465730.631)


In [22]:
cleaned_data.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 745432 entries, 0 to 745431
Data columns (total 24 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   orig_fid    745432 non-null  int64  
 1   est_soilty  745432 non-null  object 
 2   est_txcode  745426 non-null  object 
 3   zmx         745432 non-null  float64
 4   z1          745432 non-null  float64
 5   sand1       745432 non-null  int64  
 6   silt1       745432 non-null  int64  
 7   clay1       745432 non-null  int64  
 8   rock1       745432 non-null  int64  
 9   z2          359147 non-null  float64
 10  sand2       359147 non-null  float64
 11  silt2       359147 non-null  float64
 12  clay2       359147 non-null  float64
 13  rock2       359147 non-null  float64
 14  z3          19851 non-null   float64
 15  sand3       19851 non-null   float64
 16  silt3       19851 non-null   float64
 17  clay3       19851 non-null   float64
 18  rock3       19851 non-null   float64